In [ ]:
%pip install entsoe-py

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Parameters
from datetime import date, timedelta

dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")

# empty widgets = last 7 days
end_date = dbutils.widgets.get("end_date") or str(date.today())
start_date = dbutils.widgets.get("start_date") or str(date.today() - timedelta(days=7))

api_key = dbutils.secrets.get(catalog="energy", schema="configschema", key="ENTSOE_API_KEY")

# price zones: the CWE region (BE, FR, NL, DE-LU) and its neighbours in the European market coupling
PRICE_ZONES = ["BE", "FR", "NL", "DE_LU", "AT", "CH", "CZ", "PL", "DK_1", "ES"]

# borders between these zones (same list as dim_border in 02 - Creation)
BORDERS = [
    ("BE", "FR"), ("BE", "NL"), ("BE", "DE_LU"), ("FR", "DE_LU"),
    ("FR", "CH"), ("FR", "ES"), ("NL", "DE_LU"), ("NL", "DK_1"),
    ("DE_LU", "AT"), ("DE_LU", "CH"), ("DE_LU", "CZ"), ("DE_LU", "PL"),
    ("DE_LU", "DK_1"), ("AT", "CH"), ("AT", "CZ"), ("CZ", "PL"),
]

spark.conf.set("spark.sql.session.timeZone", "UTC")
print(start_date, "->", end_date)

In [ ]:
%sql
-- bronze schema and tables
USE CATALOG energy;
CREATE SCHEMA IF NOT EXISTS d2_bronze;

CREATE TABLE IF NOT EXISTS d2_bronze.bronze_prices (
  zone_code STRING,
  ts_utc TIMESTAMP,
  price_eur_mwh DOUBLE,
  ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS d2_bronze.bronze_net_position (
  zone_code STRING,
  ts_utc TIMESTAMP,
  net_position_mw DOUBLE COMMENT 'positive = export, negative = import',
  ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS d2_bronze.bronze_flows (
  from_zone STRING,
  to_zone STRING,
  ts_utc TIMESTAMP,
  flow_mw DOUBLE,
  ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS d2_bronze.bronze_scheduled (
  from_zone STRING,
  to_zone STRING,
  ts_utc TIMESTAMP,
  scheduled_mw DOUBLE COMMENT 'day-ahead commercial exchange',
  ingested_at TIMESTAMP
) USING DELTA;

In [ ]:
# ENTSO-E client
import pandas as pd
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

client = EntsoePandasClient(api_key=api_key)
start = pd.Timestamp(start_date, tz="UTC")
end = pd.Timestamp(end_date, tz="UTC")

# both directions of every border
directed_borders = BORDERS + [(b, a) for a, b in BORDERS]

In [ ]:
# write a pandas dataframe to bronze (the window is deleted first, so a re-run creates no duplicates)
def save_to_bronze(pdf, table, key):
    if pdf.empty:
        return
    pdf["ingested_at"] = pd.Timestamp.now(tz="UTC").tz_localize(None)
    spark.createDataFrame(pdf).createOrReplaceTempView("staging")
    spark.sql(f"""
        DELETE FROM {table}
        WHERE ts_utc >= '{start_date}' AND ts_utc < '{end_date}'
          AND {key} IN (SELECT DISTINCT {key} FROM staging)
    """)
    spark.sql(f"INSERT INTO {table} SELECT * FROM staging")


def to_utc(index):
    return pd.to_datetime(index, utc=True).tz_convert(None)

In [ ]:
# Day-ahead prices (bronze_prices) -- 15-min resolution since October 2025
frames = []
for zone in PRICE_ZONES:
    try:
        prices = client.query_day_ahead_prices(zone, start=start, end=end)
    except NoMatchingDataError:
        print(zone, "no data")
        continue
    frames.append(pd.DataFrame({"zone_code": zone, "ts_utc": to_utc(prices.index), "price_eur_mwh": prices.values}))
    print(zone, len(prices), "rows")

prices = pd.concat(frames).dropna()
save_to_bronze(prices[prices.ts_utc < end.tz_convert(None)], "d2_bronze.bronze_prices", "zone_code")

In [ ]:
# Day-ahead net positions (bronze_net_position)
frames = []
for zone in PRICE_ZONES:
    try:
        net_position = client.query_net_position(zone, start=start, end=end, dayahead=True)
    except NoMatchingDataError:
        print(zone, "no data")
        continue
    frames.append(pd.DataFrame({"zone_code": zone, "ts_utc": to_utc(net_position.index), "net_position_mw": net_position.values}))
    print(zone, len(net_position), "rows")

net_position = pd.concat(frames).dropna()
save_to_bronze(net_position[net_position.ts_utc < end.tz_convert(None)], "d2_bronze.bronze_net_position", "zone_code")

In [ ]:
# Cross-border physical flows (bronze_flows)
frames = []
for from_zone, to_zone in directed_borders:
    try:
        flow = client.query_crossborder_flows(from_zone, to_zone, start=start, end=end)
    except NoMatchingDataError:
        print(from_zone, "->", to_zone, "no data")
        continue
    frames.append(pd.DataFrame({"from_zone": from_zone, "to_zone": to_zone, "ts_utc": to_utc(flow.index), "flow_mw": flow.values}))

flows = pd.concat(frames).dropna()
print(len(flows), "flow rows")
save_to_bronze(flows[flows.ts_utc < end.tz_convert(None)], "d2_bronze.bronze_flows", "concat(from_zone, '>', to_zone)")

In [ ]:
# Day-ahead scheduled commercial exchanges (bronze_scheduled)
frames = []
for from_zone, to_zone in directed_borders:
    try:
        scheduled = client.query_scheduled_exchanges(from_zone, to_zone, start=start, end=end, dayahead=True)
    except NoMatchingDataError:
        print(from_zone, "->", to_zone, "no data")
        continue
    frames.append(pd.DataFrame({"from_zone": from_zone, "to_zone": to_zone, "ts_utc": to_utc(scheduled.index), "scheduled_mw": scheduled.values}))

scheduled = pd.concat(frames).dropna()
print(len(scheduled), "scheduled exchange rows")
save_to_bronze(scheduled[scheduled.ts_utc < end.tz_convert(None)], "d2_bronze.bronze_scheduled", "concat(from_zone, '>', to_zone)")

In [ ]:
%sql
-- bronze check
SELECT zone_code, COUNT(*) AS n_rows, MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts,
       ROUND(AVG(price_eur_mwh), 1) AS avg_price
FROM energy.d2_bronze.bronze_prices
GROUP BY zone_code
ORDER BY zone_code